 # Proyecto: ucuenca-sabe

 ## Dominio: 01_docentes | Capa: Bronze -> Silver



 Este notebook automatiza la carga de los archivos crudos de GTH y Analítica (PURE),

 normaliza las identificaciones mitigando la pérdida del cero a la izquierda,

 genera una auditoría de cruces (identificando registros huérfanos) y almacena el resultado

 optimizado en la capa Silver en formato Parquet.

In [1]:
# Configuración Robusta de Rutas

import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Método 1: Path absoluto (MÁS ROBUSTO)
# Obtiene la raíz del proyecto sin importar desde dónde se ejecute
def get_project_root():
    """Encuentra la raíz del proyecto buscando la carpeta 'data'"""
    current = Path.cwd()
    while current != current.parent:
        if (current / 'data').exists() and (current / 'src').exists():
            return current
        current = current.parent
    raise FileNotFoundError("❌ No se encontró la raíz del proyecto")

# Obtener raíz
PROJECT_ROOT = get_project_root()
print(f"📁 Raíz del proyecto: {PROJECT_ROOT}")

# Definir rutas
BRONZE_INTERNAS = PROJECT_ROOT / 'data' / 'bronze' / 'internas'
SILVER_DIR = PROJECT_ROOT / 'data' / 'silver'
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'

# Crear directorios si no existen
SILVER_DIR.mkdir(parents=True, exist_ok=True)
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Bronze: {BRONZE_INTERNAS}")
print(f"📂 Silver: {SILVER_DIR}")
print(f"📂 Gold: {GOLD_DIR}")

# Verificar que los archivos existen
gth_file = BRONZE_INTERNAS / "MATRIZ_ENVIADA_2.xlsx"
andat_file = BRONZE_INTERNAS / "docentes_titulo_unesco.xlsx"

print("\n🔍 Verificando archivos fuente...")
print(f"✅ GTH: {gth_file.exists()} → {gth_file}")
print(f"✅ ANALITICA: {andat_file.exists()} → {andat_file}")

📁 Raíz del proyecto: c:\Users\michu\Documents\ucuenca-sabe
📂 Bronze: c:\Users\michu\Documents\ucuenca-sabe\data\bronze\internas
📂 Silver: c:\Users\michu\Documents\ucuenca-sabe\data\silver
📂 Gold: c:\Users\michu\Documents\ucuenca-sabe\data\gold

🔍 Verificando archivos fuente...
✅ GTH: True → c:\Users\michu\Documents\ucuenca-sabe\data\bronze\internas\MATRIZ_ENVIADA_2.xlsx
✅ ANALITICA: True → c:\Users\michu\Documents\ucuenca-sabe\data\bronze\internas\docentes_titulo_unesco.xlsx


In [2]:
# ==========================================
# CELDA 2: EXPLORACIÓN BÁSICA DE GTH
# ==========================================
print("="*80)
print("📊 EXPLORACIÓN GTH - MATRIZ_ENVIADA_2")
print("="*80)

# Cargar
df_gth = pd.read_excel(gth_file, dtype=str)

# Info básica
print(f"Filas: {df_gth.shape[0]}")
print(f"Columnas: {df_gth.shape[1]}")
print(f"\nColumnas y longitud de valores:")
for col in df_gth.columns:
    nulos = df_gth[col].isnull().sum()
    # Longitud máxima y mínima de los valores
    longitudes = df_gth[col].dropna().str.len()
    if len(longitudes) > 0:
        print(f"  • {col}: min={longitudes.min()}, max={longitudes.max()}, nulos={nulos}")
    else:
        print(f"  • {col}: SIN DATOS, nulos={nulos}")

print(f"\nPrimeras 3 filas:")
display(df_gth.head(3))

📊 EXPLORACIÓN GTH - MATRIZ_ENVIADA_2
Filas: 1143
Columnas: 14

Columnas y longitud de valores:
  • CEDULA: min=10, max=10, nulos=0
  • APELLIDOS: min=6, max=26, nulos=0
  • NOMBRES: min=4, max=20, nulos=0
  • GENERO: min=1, max=1, nulos=0
  • FECHA_NACIMIENTO: min=10, max=10, nulos=0
  • TIPO_SERVIDOR: min=7, max=7, nulos=0
  • ESTADO: min=6, max=19, nulos=0
  • CARGO: min=17, max=62, nulos=0
  • FECHA_INGRESO: min=10, max=10, nulos=0
  • FECHA_TITULARIDAD: min=10, max=10, nulos=680
  • MODALIDAD_EMPLEO: min=7, max=10, nulos=0
  • CALIDAD: min=7, max=10, nulos=0
  • DEPENDENCIA: min=16, max=98, nulos=0
  • CANTON DE RESIDENCIA: min=4, max=25, nulos=556

Primeras 3 filas:


,CEDULA,APELLIDOS,NOMBRES,GENERO,FECHA_NACIMIENTO,TIPO_SERVIDOR,ESTADO,CARGO,FECHA_INGRESO,FECHA_TITULARIDAD,MODALIDAD_EMPLEO,CALIDAD,DEPENDENCIA,CANTON DE RESIDENCIA
0,0101014538,PIEDRA JARAMILLO,SANTIAGO PATRICIO,M,1955-02-27,DOCENTE,ACTIVO,PROFESOR OCASIONAL TIEMPO PARCIAL,2015-09-07,NaN,CONTRATADO,CONTRATADO,FACULTAD DE JURISPRUDENCIA Y CIENCIAS POLITICA...,NaN
1,0101224020,OCHOA MUÑOZ,JAVIER FERNANDO,M,1958-08-24,DOCENTE,ACTIVO,PROFESOR AUXILIAR TIEMPO PARCIAL NIVEL 1,2002-04-01,NaN,CONTRATADO,AUXILIAR,FACULTAD DE CIENCIAS MEDICAS,NaN
2,0101229201,PETROFF ROJAS,CESAR IVAN,M,1956-02-21,DOCENTE,ACTIVO,PROFESOR AGREGADO TIEMPO COMPLETO NIVEL 3,1982-10-01,2010-10-01,TITULAR,AGREGADO,"FACULTAD DE FILOSOFIA, LETRAS Y CIENCIAS DE LA...",NaN


In [3]:
# ==========================================
# CELDA 3: EXPLORACIÓN BÁSICA DE ANALITICA
# ==========================================
print("="*80)
print("📊 EXPLORACIÓN ANALITICA - docentes_titulo_unesco")
print("="*80)

# Cargar
df_andat = pd.read_excel(andat_file, dtype=str)

# Info básica
print(f"Filas: {df_andat.shape[0]}")
print(f"Columnas: {df_andat.shape[1]}")
print(f"\nColumnas y longitud de valores:")
for col in df_andat.columns:
    nulos = df_andat[col].isnull().sum()
    # Longitud máxima y mínima de los valores
    longitudes = df_andat[col].dropna().str.len()
    if len(longitudes) > 0:
        print(f"  • {col}: min={longitudes.min()}, max={longitudes.max()}, nulos={nulos}")
    else:
        print(f"  • {col}: SIN DATOS, nulos={nulos}")

print(f"\nPrimeras 3 filas:")
display(df_andat.head(3))

📊 EXPLORACIÓN ANALITICA - docentes_titulo_unesco
Filas: 1142
Columnas: 16

Columnas y longitud de valores:
  • NUMERO_DOCUMENTO: min=10, max=10, nulos=0
  • TIPO_DOCUMENTO: min=3, max=3, nulos=0
  • NOMBRES COMPLETOS: min=16, max=41, nulos=0
  • PAIS ORIGEN: min=4, max=25, nulos=13
  • UNIDAD ACADÉMICA: min=11, max=58, nulos=0
  • TIPO_DEDICACION: min=9, max=18, nulos=0
  • TIPO_PERSONAL: min=7, max=10, nulos=0
  • SEXO: min=5, max=6, nulos=0
  • AÑO OBTENCIÓN TITULO: min=4, max=4, nulos=8
  • PAIS ESTUDIO: min=4, max=24, nulos=8
  • NIVEL TÍTULO: min=12, max=12, nulos=8
  • TIPO TÍTULO: min=5, max=23, nulos=8
  • NOMBRE TÍTULO: min=6, max=168, nulos=8
  • CAMPO AMPLIO (UNESCO): min=9, max=53, nulos=9
  • CAMPO ESPECÍFICO (UNESCO): min=5, max=53, nulos=9
  • CAMPO DETALLADO (UNESCO): min=5, max=63, nulos=9

Primeras 3 filas:


,NUMERO_DOCUMENTO,TIPO_DOCUMENTO,NOMBRES COMPLETOS,PAIS ORIGEN,UNIDAD ACADÉMICA,TIPO_DEDICACION,TIPO_PERSONAL,SEXO,AÑO OBTENCIÓN TITULO,PAIS ESTUDIO,NIVEL TÍTULO,TIPO TÍTULO,NOMBRE TÍTULO,CAMPO AMPLIO (UNESCO),CAMPO ESPECÍFICO (UNESCO),CAMPO DETALLADO (UNESCO)
0,0102594629,CED,ASTUDILLO CORDERO JAIME SEBASTIAN,ECUADOR,FACULTAD DE ARQUITECTURA Y URBANISMO,TIEMPO COMPLETO,TITULAR,HOMBRE,2010,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN ARQUITECTURA DEL PAISAJE,"Ingeniería, industria y construcción",Arquitectura y construcción,"Arquitectura, urbanismo y restauración"
1,0102971140,CED,ATANCURI GORDILLO MANUEL ARMANDO,ECUADOR,FACULTAD DE ARQUITECTURA Y URBANISMO,TIEMPO COMPLETO,NO TITULAR,HOMBRE,2023,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN CONSTRUCCIONES MENCION EN ADMINIST...,Administración,Educación comercial y administración,Administración
2,0105161434,CED,AUQUILLA CLAVIJO SEBASTIAN FELIPE,ECUADOR,FACULTAD DE ARQUITECTURA Y URBANISMO,TIEMPO COMPLETO,NO TITULAR,HOMBRE,2024,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN CONSTRUCCIONES,"Ingeniería, industria y construcción",Arquitectura y construcción,Construcción e ingeniería civil


In [12]:
# ==========================================
# CELDA 2: CARGAR Y NORMALIZAR (UNA SOLA VEZ)
# ==========================================
print("="*80)
print("📊 CARGA Y NORMALIZACIÓN DE CÉDULAS")
print("="*80)

# Cargar
df_gth = pd.read_excel(gth_file, dtype=str)
df_andat = pd.read_excel(andat_file, dtype=str)

print(f"GTH: {df_gth.shape[0]} filas")
print(f"ANALITICA: {df_andat.shape[0]} filas")

# Columnas de cédula
col_gth = "CEDULA"
col_andat = "NUMERO_DOCUMENTO"

# FUNCIÓN ÚNICA de normalización
def normalizar_cedula(serie):
    return (serie.astype(str)
                 .str.strip()
                 .str.replace(r'[^0-9]', '', regex=True)
                 .str.zfill(10))

# Aplicar normalización UNA SOLA VEZ
df_gth['CEDULA_NORM'] = normalizar_cedula(df_gth[col_gth])
df_andat['CEDULA_NORM'] = normalizar_cedula(df_andat[col_andat])

print(f"\n✅ Cédulas normalizadas en GTH: {df_gth['CEDULA_NORM'].nunique()} únicas")
print(f"✅ Cédulas normalizadas en ANALITICA: {df_andat['CEDULA_NORM'].nunique()} únicas")

# Mostrar ejemplos
print(f"\nEjemplos GTH: {df_gth['CEDULA_NORM'].head(5).tolist()}")
print(f"Ejemplos ANALITICA: {df_andat['CEDULA_NORM'].head(5).tolist()}")

📊 CARGA Y NORMALIZACIÓN DE CÉDULAS
GTH: 1143 filas
ANALITICA: 1142 filas

✅ Cédulas normalizadas en GTH: 1143 únicas
✅ Cédulas normalizadas en ANALITICA: 1136 únicas

Ejemplos GTH: ['0101014538', '0101224020', '0101229201', '0101304129', '0101311397']
Ejemplos ANALITICA: ['0102594629', '0102971140', '0105161434', '0103567012', '0103405338']


Dado que analítica de las 1142 filas , solo 1136 son unicos y al analizar vemos que son duplicados, parecen ser- 

In [8]:
# ==========================================
# CELDA: ANÁLISIS DE DUPLICADOS EN ANALITICA
# ==========================================
print("="*80)
print("🔍 ANÁLISIS DE DUPLICADOS - ANALITICA")
print("="*80)

col_andat = "NUMERO_DOCUMENTO"

# Cargar si no está cargado
df_andat = pd.read_excel(andat_file, dtype=str)

print(f"Total filas: {len(df_andat)}")
print(f"Cédulas únicas: {df_andat[col_andat].nunique()}")
print(f"Diferencia: {len(df_andat) - df_andat[col_andat].nunique()} duplicados\n")

# Encontrar cédulas duplicadas
duplicados = df_andat[df_andat.duplicated(subset=col_andat, keep=False)]
cedulas_dup = duplicados[col_andat].unique()

print(f"Cédulas que aparecen más de una vez: {len(cedulas_dup)}")
print(f"\nCédulas duplicadas:")
for ced in cedulas_dup:
    count = (df_andat[col_andat] == ced).sum()
    print(f"  • {ced}: aparece {count} veces")

# Mostrar todas las filas duplicadas completas
print(f"\n📋 Registros duplicados completos ({len(duplicados)} filas):")
display(duplicados.sort_values(by=col_andat))

🔍 ANÁLISIS DE DUPLICADOS - ANALITICA
Total filas: 1142
Cédulas únicas: 1136
Diferencia: 6 duplicados

Cédulas que aparecen más de una vez: 6

Cédulas duplicadas:
  • 0105168785: aparece 2 veces
  • 0104907399: aparece 2 veces
  • 0102972585: aparece 2 veces
  • 0106529688: aparece 2 veces
  • 0104715297: aparece 2 veces
  • 0106488513: aparece 2 veces

📋 Registros duplicados completos (12 filas):


,NUMERO_DOCUMENTO,TIPO_DOCUMENTO,NOMBRES COMPLETOS,PAIS ORIGEN,UNIDAD ACADÉMICA,TIPO_DEDICACION,TIPO_PERSONAL,SEXO,AÑO OBTENCIÓN TITULO,PAIS ESTUDIO,NIVEL TÍTULO,TIPO TÍTULO,NOMBRE TÍTULO,CAMPO AMPLIO (UNESCO),CAMPO ESPECÍFICO (UNESCO),CAMPO DETALLADO (UNESCO)
532,0102972585,CED,NAJERA AVILEZ PRISCILA ALEXANDRA,ECUADOR,FACULTAD DE CIENCIAS MÉDICAS,TIEMPO COMPLETO,NO TITULAR,MUJER,2009,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN EDUCACION MENCION EDUCACION INFANT...,Educación,Educación,Educación
533,0102972585,CED,NAJERA AVILEZ PRISCILA ALEXANDRA,ECUADOR,FACULTAD DE CIENCIAS MÉDICAS,MEDIO TIEMPO,NO TITULAR,MUJER,2009,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN EDUCACION MENCION EDUCACION INFANT...,Educación,Educación,Educación
705,0104715297,CED,NARVAEZ VERA MONICA ALEXANDRA,ECUADOR,FACULTAD DE CIENCIAS QUÍMICAS,TIEMPO COMPLETO,NO TITULAR,MUJER,2019,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN TOXICOLOGIA INDUSTRIAL Y AMBIENTAL,Ciencias naturales matemáticas y estadísticas,Medio ambiente,Medio ambiente
706,0104715297,CED,NARVAEZ VERA MONICA ALEXANDRA,ECUADOR,FACULTAD DE CIENCIAS QUÍMICAS,TIEMPO PARCIAL,NO TITULAR,MUJER,2019,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN TOXICOLOGIA INDUSTRIAL Y AMBIENTAL,Ciencias naturales matemáticas y estadísticas,Medio ambiente,Medio ambiente
293,0104907399,CED,BOJORQUE CAMPOVERDE MILTON RODOLFO,ECUADOR,FACULTAD DE CIENCIAS ECONÓMICAS Y ADMINISTRATIVAS,POR HORAS,NO TITULAR,HOMBRE,2017,ECUADOR,TERCER NIVEL,GRADO,INGENIERO CIVIL CON ENFASIS EN GERENCIA DE CON...,"Ingeniería, industria y construcción",Arquitectura y construcción,Construcción e ingeniería civil
294,0104907399,CED,BOJORQUE CAMPOVERDE MILTON RODOLFO,ECUADOR,FACULTAD DE CIENCIAS ECONÓMICAS Y ADMINISTRATIVAS,TIEMPO PARCIAL,NO TITULAR,HOMBRE,2017,ECUADOR,TERCER NIVEL,GRADO,INGENIERO CIVIL CON ENFASIS EN GERENCIA DE CON...,"Ingeniería, industria y construcción",Arquitectura y construcción,Construcción e ingeniería civil
182,0105168785,CED,LUNA ABRIL PATRICIO JAVIER,ECUADOR,FACULTAD DE CIENCIAS AGROPECUARIAS,TIEMPO PARCIAL,NO TITULAR,HOMBRE,2024,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN HIDROLOGIA MENCION EN ECOHIDROLOGIA,"Ingeniería, industria y construcción",Arquitectura y construcción,Construcción e ingeniería civil
1131,0105168785,CED,LUNA ABRIL PATRICIO JAVIER,ECUADOR,VICERRECTORADO DE INVESTIGACIÓN,TIEMPO COMPLETO,NO TITULAR,HOMBRE,2024,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN HIDROLOGIA MENCION EN ECOHIDROLOGIA,"Ingeniería, industria y construcción",Arquitectura y construcción,Construcción e ingeniería civil
712,0106488513,CED,ORELLANA COBOS ANA BELEN,ECUADOR,FACULTAD DE CIENCIAS QUÍMICAS,TIEMPO PARCIAL,NO TITULAR,MUJER,2022,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN DIAGNOSTICO DE LABORATORIO CLINICO...,Salud y Bienestar,Salud,Medicina
824,0106488513,CED,ORELLANA COBOS ANA BELEN,ECUADOR,"FACULTAD DE FILOSOFÍA, LETRAS Y CIENCIAS DE LA...",MEDIO TIEMPO,NO TITULAR,MUJER,2022,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN DIAGNOSTICO DE LABORATORIO CLINICO...,Salud y Bienestar,Salud,Medicina


In [ ]:
# ==========================================
# CELDA 2.5: PIVOTEAR DUPLICADOS SIN PERDER COLUMNAS
# ==========================================
print("🔄 PIVOTEANDO DUPLICADOS DE ANALITICA (CONSERVANDO TODAS LAS COLUMNAS)")
print("="*80)

# Columnas que se expanden (las que varían por docente)
columnas_a_pivotear = ['UNIDAD ACADÉMICA', 'TIPO_DEDICACION']

# Columnas que NO varían (las mantenemos igual)
# Son todas menos CEDULA_NORM, orden_temp, fila_num, y las pivotadas
cols_fijas = [col for col in df_andat.columns 
              if col not in columnas_a_pivotear + ['CEDULA_NORM', 'orden_temp', 'fila_num']]

print(f"Columnas fijas (primera ocurrencia): {cols_fijas}")
print(f"Columnas a expandir: {columnas_a_pivotear}")

# 1. Ordenar para que TIEMPO COMPLETO quede primero
orden_dedicacion = {'TIEMPO COMPLETO': 0, 'MEDIO TIEMPO': 1, 'TIEMPO PARCIAL': 2, 'POR HORAS': 3}
df_andat['orden_temp'] = df_andat['TIPO_DEDICACION'].map(orden_dedicacion).fillna(99)

# 2. Ordenar por cédula y prioridad
df_andat_sorted = df_andat.sort_values(['CEDULA_NORM', 'orden_temp'])

# 3. Contador por cédula
df_andat_sorted['fila_num'] = df_andat_sorted.groupby('CEDULA_NORM').cumcount() + 1

# 4. Pivotear SOLO las columnas que varían
df_pivot = df_andat_sorted.pivot_table(
    index='CEDULA_NORM',
    columns='fila_num',
    values=columnas_a_pivotear,
    aggfunc='first'
)

# 5. Aplanar columnas pivotadas
df_pivot.columns = [f'{col[0]}_{int(col[1])}' for col in df_pivot.columns]
df_pivot = df_pivot.reset_index()

# 6. Tomar la PRIMERA fila de cada cédula para las columnas fijas
df_fijas = df_andat_sorted.groupby('CEDULA_NORM')[cols_fijas].first().reset_index()

# 7. Unir pivot + fijas
df_andat_pivot = df_pivot.merge(df_fijas, on='CEDULA_NORM', how='left')

print(f"\n✅ ANALITICA pivotado (COMPLETO):")
print(f"   Antes: {len(df_andat)} filas, {len(df_andat.columns)} columnas")
print(f"   Ahora: {len(df_andat_pivot)} filas, {len(df_andat_pivot.columns)} columnas")
print(f"\n📋 TODAS las columnas:")
for i, col in enumerate(df_andat_pivot.columns, 1):
    print(f"   {i:2d}. {col}")

# Verificar
print(f"\n🔍 Cédulas únicas: {df_andat_pivot['CEDULA_NORM'].nunique()} de {len(df_andat_pivot)} filas")
print(f"📋 Ejemplo:")
display(df_andat_pivot.head(2))

🔄 PIVOTEANDO DUPLICADOS DE ANALITICA (CONSERVANDO TODAS LAS COLUMNAS)
Columnas fijas (primera ocurrencia): ['NUMERO_DOCUMENTO', 'TIPO_DOCUMENTO', 'NOMBRES COMPLETOS', 'PAIS ORIGEN', 'TIPO_PERSONAL', 'SEXO', 'AÑO OBTENCIÓN TITULO', 'PAIS ESTUDIO', 'NIVEL TÍTULO', 'TIPO TÍTULO', 'NOMBRE TÍTULO', 'CAMPO AMPLIO (UNESCO)', 'CAMPO ESPECÍFICO (UNESCO)', 'CAMPO DETALLADO (UNESCO)']
Columnas a expandir: ['UNIDAD ACADÉMICA', 'TIPO_DEDICACION']

✅ ANALITICA pivotado (COMPLETO):
   Antes: 1142 filas, 18 columnas
   Ahora: 1136 filas, 19 columnas

📋 TODAS las columnas:
    1. CEDULA_NORM
    2. TIPO_DEDICACION_1
    3. TIPO_DEDICACION_2
    4. UNIDAD ACADÉMICA_1
    5. UNIDAD ACADÉMICA_2
    6. NUMERO_DOCUMENTO
    7. TIPO_DOCUMENTO
    8. NOMBRES COMPLETOS
    9. PAIS ORIGEN
   10. TIPO_PERSONAL
   11. SEXO
   12. AÑO OBTENCIÓN TITULO
   13. PAIS ESTUDIO
   14. NIVEL TÍTULO
   15. TIPO TÍTULO
   16. NOMBRE TÍTULO
   17. CAMPO AMPLIO (UNESCO)
   18. CAMPO ESPECÍFICO (UNESCO)
   19. CAMPO DETALLADO 

,CEDULA_NORM,TIPO_DEDICACION_1,TIPO_DEDICACION_2,UNIDAD ACADÉMICA_1,UNIDAD ACADÉMICA_2,NUMERO_DOCUMENTO,TIPO_DOCUMENTO,NOMBRES COMPLETOS,PAIS ORIGEN,TIPO_PERSONAL,SEXO,AÑO OBTENCIÓN TITULO,PAIS ESTUDIO,NIVEL TÍTULO,TIPO TÍTULO,NOMBRE TÍTULO,CAMPO AMPLIO (UNESCO),CAMPO ESPECÍFICO (UNESCO),CAMPO DETALLADO (UNESCO)
0,0101014538,TIEMPO PARCIAL,NaN,FACULTAD DE JURISPRUDENCIA Y CIENCIAS POLÍTICA...,NaN,0101014538,CED,PIEDRA JARAMILLO SANTIAGO PATRICIO,ECUADOR,NO TITULAR,HOMBRE,2015,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN MEDICINA FORENSE,Salud y Bienestar,Salud,Medicina
1,0101059863,POR HORAS,NaN,FACULTAD DE JURISPRUDENCIA Y CIENCIAS POLÍTICA...,NaN,0101059863,CED,CASTRO RIERA CARLOS MANUEL,ECUADOR,NO TITULAR,HOMBRE,2012,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN DERECHO ADMINISTRATIVO,"Ciencias sociales periodismo, información y de...",Derecho,Derecho


In [20]:
# CELDA 3: CHEQUEO DE MATCH (con df_andat_pivot COMPLETO)

print(" CHEQUEO DE MATCH")


set_gth = set(df_gth['CEDULA_NORM'].dropna())
set_andat = set(df_andat_pivot['CEDULA_NORM'].dropna())  # ← USAR df_andat_pivot

ambas = set_gth & set_andat
solo_gth = set_gth - set_andat
solo_andat = set_andat - set_gth

print(f"""
📊 RESULTADOS:
   ✅ En ambas bases:  {len(ambas)} docentes
   ⚠️ Solo en GTH:     {len(solo_gth)} docentes
   ⚠️ Solo en ANALITICA: {len(solo_andat)} docentes
   ─────────────────────────────
   📁 Total GTH:       {len(set_gth)} cédulas únicas
   📁 Total ANALITICA: {len(set_andat)} cédulas únicas
""")

 CHEQUEO DE MATCH

📊 RESULTADOS:
   ✅ En ambas bases:  1104 docentes
   ⚠️ Solo en GTH:     39 docentes
   ⚠️ Solo en ANALITICA: 32 docentes
   ─────────────────────────────
   📁 Total GTH:       1143 cédulas únicas
   📁 Total ANALITICA: 1136 cédulas únicas



In [21]:
# ==========================================
# CELDA 4: OUTER JOIN (usando df_andat_pivot)
# ==========================================
print("="*80)
print("🔗 OUTER JOIN: CONSERVANDO TODOS LOS DOCENTES")
print("="*80)

# OUTER JOIN con la versión pivotada
df_full = df_gth.merge(
    df_andat_pivot,
    how="outer",
    on='CEDULA_NORM',
    indicator=True,
    suffixes=("_GTH", "_ANDAT")
)

print(f"\n📊 RESULTADO DEL OUTER JOIN:")
print(f"   Total docentes: {len(df_full)}")
print(f"\n   ✅ both:        {(df_full['_merge']=='both').sum()} (en ambas bases)")
print(f"   ⚠️ left_only:    {(df_full['_merge']=='left_only').sum()} (solo GTH)")
print(f"   ⚠️ right_only:   {(df_full['_merge']=='right_only').sum()} (solo ANALITICA)")

print(f"\n📋 Primeros 3 registros:")
display(df_full.head(3))

🔗 OUTER JOIN: CONSERVANDO TODOS LOS DOCENTES

📊 RESULTADO DEL OUTER JOIN:
   Total docentes: 1175

   ✅ both:        1104 (en ambas bases)
   ⚠️ left_only:    39 (solo GTH)
   ⚠️ right_only:   32 (solo ANALITICA)

📋 Primeros 3 registros:


,CEDULA,APELLIDOS,NOMBRES,GENERO,FECHA_NACIMIENTO,TIPO_SERVIDOR,ESTADO,CARGO,FECHA_INGRESO,FECHA_TITULARIDAD,...,SEXO,AÑO OBTENCIÓN TITULO,PAIS ESTUDIO,NIVEL TÍTULO,TIPO TÍTULO,NOMBRE TÍTULO,CAMPO AMPLIO (UNESCO),CAMPO ESPECÍFICO (UNESCO),CAMPO DETALLADO (UNESCO),_merge
0,0101014538,PIEDRA JARAMILLO,SANTIAGO PATRICIO,M,1955-02-27,DOCENTE,ACTIVO,PROFESOR OCASIONAL TIEMPO PARCIAL,2015-09-07,NaN,...,HOMBRE,2015,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN MEDICINA FORENSE,Salud y Bienestar,Salud,Medicina,both
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,HOMBRE,2012,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN DERECHO ADMINISTRATIVO,"Ciencias sociales periodismo, información y de...",Derecho,Derecho,right_only
2,0101224020,OCHOA MUÑOZ,JAVIER FERNANDO,M,1958-08-24,DOCENTE,ACTIVO,PROFESOR AUXILIAR TIEMPO PARCIAL NIVEL 1,2002-04-01,NaN,...,HOMBRE,2014,ESPAÑA,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MASTER SOBRE INFECCION POR EL VIH,Salud y Bienestar,Salud,Medicina,both


In [24]:
# ==========================================
# CELDA 5: FILTRAR SOLO MATCH (BOTH) + LIMPIAR COLUMNAS
# ==========================================
print("="*80)
print("✅ EXTRAYENDO SOLO DOCENTES CON MATCH")
print("="*80)

# Filtrar solo los que están en AMBAS bases
df_match = df_full[df_full['_merge'] == 'both'].copy()

# Eliminar columna _merge
df_match = df_match.drop(columns=['_merge'])

# Identificar columnas de cédula a eliminar
columnas_cedula_extra = ['CEDULA', 'CEDULA_GTH', 'CEDULA_ANDAT', 'TIPO_DOCUMENTO', 'NUMERO_DOCUMENTO', 'NUMERO_DOCUMENTO_GTH', 'NUMERO_DOCUMENTO_ANDAT']

# Ver cuáles existen realmente en df_match
columnas_a_eliminar = [col for col in columnas_cedula_extra if col in df_match.columns]

print(f"\n🗑️ Eliminando columnas de cédula redundantes:")
for col in columnas_a_eliminar:
    print(f"   • {col}")

# Eliminar esas columnas
df_match = df_match.drop(columns=columnas_a_eliminar, errors='ignore')

print(f"\n📊 MATCH (both) - LIMPIO:")
print(f"   Total: {len(df_match)} docentes")
print(f"   Columnas: {len(df_match.columns)}")

# Verificar que CEDULA_NORM está presente
if 'CEDULA_NORM' in df_match.columns:
    print(f"   ✅ CEDULA_NORM conservada")
else:
    print(f"   ❌ CEDULA_NORM no encontrada")

# Verificar que no hay duplicados
print(f"\n🔍 Verificación:")
print(f"   Cédulas únicas: {df_match['CEDULA_NORM'].nunique()}")
print(f"   Filas totales: {len(df_match)}")
print(f"   Duplicados: {len(df_match) - df_match['CEDULA_NORM'].nunique()}")

# Mostrar TODAS las columnas finales
print(f"\n📋 Columnas finales ({len(df_match.columns)}):")
for i, col in enumerate(df_match.columns, 1):
    print(f"   {i:2d}. {col}")

# Mostrar primeras filas
print(f"\n📋 Primeros 3 registros:")
display(df_match.head(3))

# Info rápida de nulos
print(f"\n📊 Nulos por columna:")
nulos = df_match.isnull().sum()
nulos_pct = (nulos / len(df_match)) * 100
for col in df_match.columns:
    if nulos[col] > 0:
        print(f"   • {col}: {nulos[col]} nulos ({nulos_pct[col]:.1f}%)")

✅ EXTRAYENDO SOLO DOCENTES CON MATCH

🗑️ Eliminando columnas de cédula redundantes:
   • CEDULA
   • TIPO_DOCUMENTO
   • NUMERO_DOCUMENTO

📊 MATCH (both) - LIMPIO:
   Total: 1104 docentes
   Columnas: 30
   ✅ CEDULA_NORM conservada

🔍 Verificación:
   Cédulas únicas: 1104
   Filas totales: 1104
   Duplicados: 0

📋 Columnas finales (30):
    1. APELLIDOS
    2. NOMBRES
    3. GENERO
    4. FECHA_NACIMIENTO
    5. TIPO_SERVIDOR
    6. ESTADO
    7. CARGO
    8. FECHA_INGRESO
    9. FECHA_TITULARIDAD
   10. MODALIDAD_EMPLEO
   11. CALIDAD
   12. DEPENDENCIA
   13. CANTON DE RESIDENCIA
   14. CEDULA_NORM
   15. TIPO_DEDICACION_1
   16. TIPO_DEDICACION_2
   17. UNIDAD ACADÉMICA_1
   18. UNIDAD ACADÉMICA_2
   19. NOMBRES COMPLETOS
   20. PAIS ORIGEN
   21. TIPO_PERSONAL
   22. SEXO
   23. AÑO OBTENCIÓN TITULO
   24. PAIS ESTUDIO
   25. NIVEL TÍTULO
   26. TIPO TÍTULO
   27. NOMBRE TÍTULO
   28. CAMPO AMPLIO (UNESCO)
   29. CAMPO ESPECÍFICO (UNESCO)
   30. CAMPO DETALLADO (UNESCO)

📋 Primeros

,APELLIDOS,NOMBRES,GENERO,FECHA_NACIMIENTO,TIPO_SERVIDOR,ESTADO,CARGO,FECHA_INGRESO,FECHA_TITULARIDAD,MODALIDAD_EMPLEO,...,TIPO_PERSONAL,SEXO,AÑO OBTENCIÓN TITULO,PAIS ESTUDIO,NIVEL TÍTULO,TIPO TÍTULO,NOMBRE TÍTULO,CAMPO AMPLIO (UNESCO),CAMPO ESPECÍFICO (UNESCO),CAMPO DETALLADO (UNESCO)
0,PIEDRA JARAMILLO,SANTIAGO PATRICIO,M,1955-02-27,DOCENTE,ACTIVO,PROFESOR OCASIONAL TIEMPO PARCIAL,2015-09-07,NaN,CONTRATADO,...,NO TITULAR,HOMBRE,2015,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN MEDICINA FORENSE,Salud y Bienestar,Salud,Medicina
2,OCHOA MUÑOZ,JAVIER FERNANDO,M,1958-08-24,DOCENTE,ACTIVO,PROFESOR AUXILIAR TIEMPO PARCIAL NIVEL 1,2002-04-01,NaN,CONTRATADO,...,NO TITULAR,HOMBRE,2014,ESPAÑA,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MASTER SOBRE INFECCION POR EL VIH,Salud y Bienestar,Salud,Medicina
3,PETROFF ROJAS,CESAR IVAN,M,1956-02-21,DOCENTE,ACTIVO,PROFESOR AGREGADO TIEMPO COMPLETO NIVEL 3,1982-10-01,2010-10-01,TITULAR,...,TITULAR,HOMBRE,2009,ECUADOR,CUARTO NIVEL,MAESTRÍA O EQUIVALENTE,MAGISTER EN DOCENCIA Y CURRICULO PARA LA EDUCA...,Educación,Educación,Educación



📊 Nulos por columna:
   • FECHA_TITULARIDAD: 652 nulos (59.1%)
   • CANTON DE RESIDENCIA: 532 nulos (48.2%)
   • TIPO_DEDICACION_2: 1098 nulos (99.5%)
   • UNIDAD ACADÉMICA_2: 1098 nulos (99.5%)
   • PAIS ORIGEN: 9 nulos (0.8%)
   • AÑO OBTENCIÓN TITULO: 8 nulos (0.7%)
   • PAIS ESTUDIO: 8 nulos (0.7%)
   • NIVEL TÍTULO: 8 nulos (0.7%)
   • TIPO TÍTULO: 8 nulos (0.7%)
   • NOMBRE TÍTULO: 8 nulos (0.7%)
   • CAMPO AMPLIO (UNESCO): 8 nulos (0.7%)
   • CAMPO ESPECÍFICO (UNESCO): 8 nulos (0.7%)
   • CAMPO DETALLADO (UNESCO): 8 nulos (0.7%)
